# Loan Default Prediction - Exploratory Data Analysis

This notebook explores `loan_processed_data.csv` for a loan default prediction problem.

Main goals:

- Understand columns, values, distributions, and class balance.
- Check correlations and associations against the target.
- Identify target leakage, especially post-origination repayment fields.
- Check whether the dataset supports time-aware validation.
- Prepare a practical modeling plan.

## Key Modeling Notes

**Do we need one-hot encoding before correlation analysis?**

For numeric variables, no. We can directly compute correlation against the binary target. This is equivalent to a point-biserial relationship when the target is encoded as `0/1`.

For categorical variables, normal Pearson correlation is not meaningful on raw strings. Useful options are:

- Grouped default rates by category.
- One-hot encode categories and correlate each dummy variable with the target.
- Cramer's V or chi-square tests.
- Mutual information.

One-hot correlation is useful for finding suspicious category-level leakage, but it can be noisy for rare categories and expensive for high-cardinality columns such as `emp_title`.

**Are we sure the dataset is time-aware?**

The dataset contains date-like columns such as `issue_d`, `earliest_cr_line`, `last_pymnt_d`, and `last_credit_pull_d`. That means we can perform time-aware analysis. However, only `issue_d` and transformed versions of pre-loan dates should be considered for an origination-time prediction model. Fields like `last_pymnt_d`, `last_credit_pull_d`, and repayment totals are known after the loan outcome develops and are leakage.

We should also inspect default rates by issue vintage. If recent vintages have unusually low defaults, that may indicate censoring or that the dataset has been filtered to completed loans in a way that affects time comparisons.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

DATA_PATH = Path("loan_processed_data.csv")
assert DATA_PATH.exists(), f"Missing file: {DATA_PATH.resolve()}"

df = pd.read_csv(DATA_PATH, low_memory=False, skipinitialspace=True)
df.shape

In [ ]:
df.head()

In [ ]:
overview = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": df.isna().mean().mul(100),
    "nunique": df.nunique(dropna=True),
})
overview.sort_values(["missing_pct", "nunique"], ascending=[False, False])

## Target Definition

The target column is `loan_status`. Here we map:

- `Fully Paid` -> `0`
- `Charged Off` -> `1`

This makes `1` the event of interest: default / charged off.

In [ ]:
target_counts = df["loan_status"].value_counts(dropna=False).to_frame("count")
target_counts["pct"] = target_counts["count"] / len(df) * 100
target_counts

In [ ]:
df["target_default"] = (df["loan_status"] == "Charged Off").astype(int)
df["target_default"].mean()

## Leakage Checklist

For a realistic loan-origination model, a feature must be known at the time the loan is issued. Columns observed after loan issuance can leak the answer.

Likely leakage columns in this dataset:

- `total_pymnt`
- `total_rec_int`
- `total_rec_late_fee`
- `recoveries`
- `last_pymnt_d`
- `last_pymnt_amnt`
- `last_credit_pull_d`
- `last_fico_range_high`
- `debt_settlement_flag`

`loan_status` is the target itself and must not be used as a feature.

In [ ]:
known_leakage_cols = [
    "loan_status",
    "total_pymnt",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "last_credit_pull_d",
    "last_fico_range_high",
    "debt_settlement_flag",
]

[c for c in known_leakage_cols if c in df.columns]

## Numeric Correlation Against Target

This is a quick way to identify strong numeric predictors and possible leakage. Very high correlations with the target, especially for columns that are not known at origination, deserve suspicion.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.drop("target_default", errors="ignore")

numeric_corr = (
    df[numeric_cols]
    .corrwith(df["target_default"])
    .rename("corr_with_default")
    .to_frame()
)
numeric_corr["abs_corr"] = numeric_corr["corr_with_default"].abs()
numeric_corr["leakage_candidate"] = numeric_corr.index.isin(known_leakage_cols)
numeric_corr.sort_values("abs_corr", ascending=False).head(40)

In [ ]:
top_numeric_corr = numeric_corr.sort_values("abs_corr", ascending=False).head(25).sort_values("abs_corr")

plt.figure(figsize=(9, 8))
colors = np.where(top_numeric_corr["leakage_candidate"], "tab:red", "tab:blue")
plt.barh(top_numeric_corr.index, top_numeric_corr["corr_with_default"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("Top numeric correlations with default target")
plt.xlabel("Pearson correlation with target_default")
plt.tight_layout()
plt.show()

## Categorical Associations Against Target

Raw categorical strings cannot be directly correlated with the target. First, inspect default rates by category. Then, for manageable categorical variables, one-hot encode and correlate each category dummy with the target.

In [ ]:
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
cat_summary = pd.DataFrame({
    "nunique": df[cat_cols].nunique(dropna=True),
    "missing_pct": df[cat_cols].isna().mean().mul(100),
}).sort_values("nunique", ascending=False)
cat_summary

In [ ]:
def default_rate_by_category(data, col, min_count=100):
    out = (
        data.groupby(col, dropna=False)["target_default"]
        .agg(count="count", default_rate="mean")
        .reset_index()
    )
    out["default_rate_pct"] = out["default_rate"] * 100
    out = out[out["count"] >= min_count].sort_values("default_rate", ascending=False)
    return out

for col in ["grade", "sub_grade", "term", "home_ownership", "verification_status", "purpose", "application_type", "initial_list_status", "disbursement_method", "debt_settlement_flag"]:
    if col in df.columns:
        display(default_rate_by_category(df, col, min_count=1).head(30).style.format({"default_rate": "{:.3f}", "default_rate_pct": "{:.2f}"}))

In [ ]:
# One-hot correlation for low/medium-cardinality categoricals.
# Avoid very high-cardinality columns here because they create many sparse columns and noisy rare-category signals.
max_categories_for_ohe_corr = 100
ohe_cat_cols = [
    c for c in cat_cols
    if c not in ["loan_status"] and df[c].nunique(dropna=True) <= max_categories_for_ohe_corr
]

cat_dummies = pd.get_dummies(df[ohe_cat_cols], dummy_na=True, drop_first=False, dtype="uint8")
dummy_corr = cat_dummies.corrwith(df["target_default"]).rename("corr_with_default").to_frame()
dummy_corr["abs_corr"] = dummy_corr["corr_with_default"].abs()
dummy_corr["parent_col"] = dummy_corr.index.to_series().str.extract("^(.*?)_")[0].values
dummy_corr.sort_values("abs_corr", ascending=False).head(50)

## Mutual Information Screening

Mutual information can capture nonlinear relationships. This section uses ordinal/category codes for a broad screening pass. Treat this as ranking, not as a final model explanation.

In [ ]:
try:
    from sklearn.feature_selection import mutual_info_classif

    leakage_for_screening = [c for c in known_leakage_cols if c in df.columns]
    screen_cols = [c for c in df.columns if c not in leakage_for_screening + ["target_default"]]
    sample = df[screen_cols + ["target_default"]].sample(min(100_000, len(df)), random_state=42)

    X_mi = sample[screen_cols].copy()
    discrete_mask = []
    for c in X_mi.columns:
        if X_mi[c].dtype == "object":
            X_mi[c] = X_mi[c].astype("category").cat.codes
            discrete_mask.append(True)
        else:
            X_mi[c] = X_mi[c].fillna(X_mi[c].median())
            discrete_mask.append(False)

    mi = mutual_info_classif(X_mi, sample["target_default"], discrete_features=discrete_mask, random_state=42)
    mi_table = pd.DataFrame({"feature": screen_cols, "mutual_info": mi}).sort_values("mutual_info", ascending=False)
    display(mi_table.head(40))
except Exception as exc:
    print(f"Skipping mutual information because sklearn is unavailable or errored: {exc}")

## Numeric Distributions and Outliers

In [ ]:
selected_numeric = [
    "loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_high",
    "inq_last_6mths", "open_acc", "pub_rec", "revol_bal", "revol_util",
    "total_acc", "tot_cur_bal", "acc_open_past_24mths", "bc_open_to_buy",
    "mort_acc", "mths_since_recent_inq", "pct_tl_nvr_dlq", "total_bal_ex_mort",
]
selected_numeric = [c for c in selected_numeric if c in df.columns]

df[selected_numeric].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T

In [ ]:
quality_checks = pd.Series({
    "annual_inc <= 0": (df["annual_inc"] <= 0).sum() if "annual_inc" in df else np.nan,
    "dti < 0": (df["dti"] < 0).sum() if "dti" in df else np.nan,
    "dti > 100": (df["dti"] > 100).sum() if "dti" in df else np.nan,
    "revol_util > 100": (df["revol_util"] > 100).sum() if "revol_util" in df else np.nan,
    "last_fico_range_high == 0": (df["last_fico_range_high"] == 0).sum() if "last_fico_range_high" in df else np.nan,
})
quality_checks.to_frame("row_count")

In [ ]:
plot_cols = ["loan_amnt", "int_rate", "annual_inc", "dti", "fico_range_high", "revol_util", "tot_cur_bal"]
plot_cols = [c for c in plot_cols if c in df.columns]

fig, axes = plt.subplots(len(plot_cols), 1, figsize=(10, 3 * len(plot_cols)))
if len(plot_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, plot_cols):
    upper = df[col].quantile(0.99)
    df.loc[df[col] <= upper, col].hist(bins=50, ax=ax)
    ax.set_title(f"{col} distribution, clipped at 99th percentile for plotting")
plt.tight_layout()
plt.show()

## Time Awareness and Vintage Analysis

`issue_d` is the most important date for validation. A random train/test split can overstate performance because it mixes older and newer loan vintages. A better validation setup is:

- Train on older issue dates.
- Validate/test on later issue dates.

Watch for target censoring: recent loans may not have had enough time to default, especially 60-month loans.

In [ ]:
df["issue_date"] = pd.to_datetime(df["issue_d"], format="%b-%Y", errors="coerce")
df["earliest_cr_date"] = pd.to_datetime(df["earliest_cr_line"], format="%b-%Y", errors="coerce")
df["last_payment_date"] = pd.to_datetime(df["last_pymnt_d"], format="%b-%Y", errors="coerce")
df["last_credit_pull_date"] = pd.to_datetime(df["last_credit_pull_d"], format="%b-%Y", errors="coerce")

date_ranges = pd.DataFrame({
    "min": df[["issue_date", "earliest_cr_date", "last_payment_date", "last_credit_pull_date"]].min(),
    "max": df[["issue_date", "earliest_cr_date", "last_payment_date", "last_credit_pull_date"]].max(),
    "missing": df[["issue_date", "earliest_cr_date", "last_payment_date", "last_credit_pull_date"]].isna().sum(),
})
date_ranges

In [ ]:
vintage = (
    df.assign(issue_year=df["issue_date"].dt.year, issue_month=df["issue_date"].dt.to_period("M").astype(str))
    .groupby("issue_year")["target_default"]
    .agg(count="count", default_rate="mean")
)
vintage["default_rate_pct"] = vintage["default_rate"] * 100
vintage

In [ ]:
monthly_vintage = (
    df.assign(issue_month=df["issue_date"].dt.to_period("M").astype(str))
    .groupby("issue_month")["target_default"]
    .agg(count="count", default_rate="mean")
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(14, 5))
ax1.plot(monthly_vintage["issue_month"], monthly_vintage["default_rate"] * 100, marker="o", linewidth=1)
ax1.set_ylabel("Default rate (%)")
ax1.set_title("Default rate by issue month")
ax1.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Credit history age is a valid feature if derived from dates known at origination.
df["credit_history_months_at_issue"] = (
    (df["issue_date"].dt.year - df["earliest_cr_date"].dt.year) * 12
    + (df["issue_date"].dt.month - df["earliest_cr_date"].dt.month)
)

df[["issue_d", "earliest_cr_line", "credit_history_months_at_issue"]].head()

In [ ]:
# A simple maturity/censoring check: compare last payment timing against issue timing.
# This uses post-origination data only for diagnostics, not for model features.
df["months_issue_to_last_payment"] = (
    (df["last_payment_date"].dt.year - df["issue_date"].dt.year) * 12
    + (df["last_payment_date"].dt.month - df["issue_date"].dt.month)
)

maturity_diag = (
    df.assign(issue_year=df["issue_date"].dt.year)
    .groupby(["issue_year", "term"])["months_issue_to_last_payment"]
    .agg(count="count", median_months="median", p10=lambda s: s.quantile(0.10), p90=lambda s: s.quantile(0.90))
)
maturity_diag

## Candidate Feature Sets

Start with a conservative feature set that only includes variables plausibly known at loan origination. Then iterate.

In [ ]:
drop_for_origination_model = [c for c in known_leakage_cols if c in df.columns]

# High-cardinality or redundant text-like fields. Keep them out of the first model.
drop_initially = [c for c in ["emp_title", "title", "zip_code"] if c in df.columns]

candidate_features = [
    c for c in df.columns
    if c not in drop_for_origination_model
    and c not in drop_initially
    and c not in ["target_default", "issue_date", "earliest_cr_date", "last_payment_date", "last_credit_pull_date", "months_issue_to_last_payment"]
]

candidate_features

## What Else To Consider

- **Leakage discipline:** decide whether the model predicts at origination, during servicing, or after delinquency. The allowed features differ for each use case.
- **Time split:** prefer validation by `issue_d` over random split.
- **Censoring:** recent loans may not have had enough time to default. Inspect issue vintages before trusting recent rows.
- **Class imbalance:** the positive class is about 20%, so imbalance exists but is not extreme. Use ROC-AUC, PR-AUC, recall at fixed precision, and calibration rather than accuracy alone.
- **Calibration:** credit-risk models often need reliable probabilities, not just rankings.
- **Threshold selection:** choose thresholds based on business cost: false approval vs false rejection.
- **Fairness and compliance:** avoid or carefully govern location/proxy variables such as `addr_state` and `zip_code` if the model is for real credit decisions.
- **Feature stability:** check whether relationships shift over issue years.
- **Interpretability:** logistic regression and monotonic gradient boosting can be useful baselines before more complex models.
- **Rare categories:** high-cardinality fields can overfit. Use grouping, hashing, target encoding with cross-validation, or drop them initially.

## Suggested Next Step

Build two baselines:

1. **Clean origination model:** no leakage columns, time-based split, simple one-hot encoding, logistic regression or gradient boosting.
2. **Leakage demonstration model:** include leakage columns intentionally once, just to show how inflated the metrics become.

The difference between those two models is usually the fastest way to prove why leakage handling matters.

# Origination Modeling Workflow

This section extends the EDA into a credit-risk modeling workflow for account-opening decisioning.

The goal is to predict default risk using only features that would plausibly be available at origination. Post-origination repayment, servicing, collection, and settlement fields are excluded from the clean model.

We will build and compare three feature views:

1. **Clean origination-only model**: the main candidate model.
2. **Pricing/underwriting benchmark**: includes `int_rate`, `grade`, and `sub_grade` to show how much signal is embedded in lender decision/pricing outputs.
3. **Leakage demonstration**: intentionally includes post-origination fields to show why leakage controls matter. This model is not valid for deployment.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42

# Keep this as None for a full run. Set to an integer, e.g. 250_000, for a faster demo iteration.
MODEL_SAMPLE_SIZE = None


## Origination-Safe Feature Preparation

The helper below performs feature engineering and creates feature sets with different leakage assumptions.

Important framing:

- `include_pricing_outputs=False` is the primary account-opening setup.
- `include_pricing_outputs=True` is a diagnostic benchmark only unless the business confirms those fields are available before this model would be used.
- `include_leakage=True` is only for demonstrating invalid performance inflation.


In [ ]:
TARGET_COL = "loan_status"
TARGET_POSITIVE_LABEL = "Charged Off"

POST_ORIGINATION_LEAKAGE_COLS = [
    "total_pymnt",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "last_pymnt_d",
    "last_pymnt_amnt",
    "last_credit_pull_d",
    "last_fico_range_high",
    "debt_settlement_flag",
]

PRICING_OUTPUT_COLS = ["int_rate", "grade", "sub_grade"]
HIGH_CARDINALITY_DROP_COLS = ["emp_title", "title", "zip_code"]
DATE_HELPER_COLS = [
    "issue_date",
    "earliest_cr_date",
    "last_payment_date",
    "last_credit_pull_date",
    "months_issue_to_last_payment",
]

EMP_LENGTH_MAP = {
    "< 1 year": 0,
    "1 year": 1,
    "2 years": 2,
    "3 years": 3,
    "4 years": 4,
    "5 years": 5,
    "6 years": 6,
    "7 years": 7,
    "8 years": 8,
    "9 years": 9,
    "10+ years": 10,
}


def add_origination_features(data):
    out = data.copy()

    out["target_default"] = (out[TARGET_COL] == TARGET_POSITIVE_LABEL).astype(int)

    out["issue_date"] = pd.to_datetime(out["issue_d"].astype(str).str.strip(), format="%b-%Y", errors="coerce")
    out["earliest_cr_date"] = pd.to_datetime(out["earliest_cr_line"].astype(str).str.strip(), format="%b-%Y", errors="coerce")

    out["term_months"] = out["term"].astype(str).str.extract(r"(\d+)").astype(float)
    out["emp_length_num"] = out["emp_length"].map(EMP_LENGTH_MAP).astype(float)

    out["credit_history_months_at_issue"] = (
        (out["issue_date"].dt.year - out["earliest_cr_date"].dt.year) * 12
        + (out["issue_date"].dt.month - out["earliest_cr_date"].dt.month)
    )

    annual_inc_safe = out["annual_inc"].replace(0, np.nan)
    out["loan_to_income"] = out["loan_amnt"] / annual_inc_safe
    out["revol_bal_to_income"] = out["revol_bal"] / annual_inc_safe
    out["open_acc_to_total_acc"] = out["open_acc"] / out["total_acc"].replace(0, np.nan)

    return out


def build_feature_view(data, include_pricing_outputs=False, include_leakage=False):
    modeled = add_origination_features(data)
    y = modeled["target_default"]

    drop_cols = [TARGET_COL, "target_default"] + DATE_HELPER_COLS
    drop_cols += ["issue_d", "earliest_cr_line", "term", "emp_length"]
    drop_cols += [c for c in HIGH_CARDINALITY_DROP_COLS if c in modeled.columns]

    if not include_pricing_outputs:
        drop_cols += [c for c in PRICING_OUTPUT_COLS if c in modeled.columns]

    if not include_leakage:
        drop_cols += [c for c in POST_ORIGINATION_LEAKAGE_COLS if c in modeled.columns]

    X = modeled.drop(columns=[c for c in drop_cols if c in modeled.columns])
    split_dates = modeled["issue_date"]

    return X, y, split_dates, modeled


## Chronological Train / Validation / Test Split

The final evidence should be time-aware. This split uses older vintages for training and later vintages for validation/test.

If the most recent year appears immature or censored, present that clearly and consider treating it as a stress/generalization period rather than the only final test.


In [ ]:
def chronological_split(X, y, split_dates, train_end="2016-12-31", valid_end="2017-12-31"):
    train_end = pd.Timestamp(train_end)
    valid_end = pd.Timestamp(valid_end)

    train_mask = split_dates <= train_end
    valid_mask = (split_dates > train_end) & (split_dates <= valid_end)
    test_mask = split_dates > valid_end

    splits = {
        "train": (X.loc[train_mask], y.loc[train_mask]),
        "valid": (X.loc[valid_mask], y.loc[valid_mask]),
        "test": (X.loc[test_mask], y.loc[test_mask]),
    }

    split_summary = pd.DataFrame({
        name: {
            "rows": len(split_y),
            "default_rate": split_y.mean(),
            "min_issue_date": split_dates.loc[split_y.index].min(),
            "max_issue_date": split_dates.loc[split_y.index].max(),
        }
        for name, (_, split_y) in splits.items()
    }).T

    return splits, split_summary


X_clean, y, split_dates, modeled_df = build_feature_view(
    df,
    include_pricing_outputs=False,
    include_leakage=False,
)

if MODEL_SAMPLE_SIZE is not None and MODEL_SAMPLE_SIZE < len(X_clean):
    sample_idx = X_clean.sample(MODEL_SAMPLE_SIZE, random_state=RANDOM_STATE).index
    X_clean = X_clean.loc[sample_idx]
    y = y.loc[sample_idx]
    split_dates = split_dates.loc[sample_idx]

splits, split_summary = chronological_split(X_clean, y, split_dates)
split_summary


## Preprocessing Pipelines

The Logistic Regression pipeline uses one-hot encoding and scaling. The tree/boosting pipeline uses imputation plus ordinal encoding for categoricals.

Both pipelines fit preprocessing only on the training data, which prevents validation/test leakage through encoders or scalers.


In [ ]:
def get_feature_types(X):
    numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    return numeric_features, categorical_features


def make_logistic_pipeline(X):
    numeric_features, categorical_features = get_feature_types(X)

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=True)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ],
        remainder="drop",
    )

    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="saga",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])


def make_hgb_pipeline(X):
    numeric_features, categorical_features = get_feature_types(X)

    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, numeric_features),
            ("cat", categorical_pipe, categorical_features),
        ],
        remainder="drop",
    )

    return Pipeline(steps=[
        ("preprocess", preprocessor),
        ("model", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.06,
            max_leaf_nodes=31,
            l2_regularization=0.05,
            class_weight="balanced",
            random_state=RANDOM_STATE,
        )),
    ])


## Train Baseline Models

Start with the clean origination-only feature view. Keep the benchmark and leakage-demo models separate so the final recommendation is not accidentally based on invalid features.


In [ ]:
X_train, y_train = splits["train"]
X_valid, y_valid = splits["valid"]
X_test, y_test = splits["test"]

logistic_model = make_logistic_pipeline(X_train)
hgb_model = make_hgb_pipeline(X_train)

logistic_model.fit(X_train, y_train)
hgb_model.fit(X_train, y_train)

"Models trained"


## Evaluation Helpers

Use ranking metrics, threshold metrics, and calibration metrics. Accuracy is not enough for credit-risk decisioning.


In [ ]:
def score_model(model, X_eval, y_eval, model_name, split_name, threshold=0.5):
    proba = model.predict_proba(X_eval)[:, 1]
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_eval, pred).ravel()

    return {
        "model": model_name,
        "split": split_name,
        "threshold": threshold,
        "roc_auc": roc_auc_score(y_eval, proba),
        "pr_auc": average_precision_score(y_eval, proba),
        "brier_score": brier_score_loss(y_eval, proba),
        "precision": precision_score(y_eval, pred, zero_division=0),
        "recall": recall_score(y_eval, pred, zero_division=0),
        "f1": f1_score(y_eval, pred, zero_division=0),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def evaluate_models(models, datasets, threshold=0.5):
    rows = []
    for model_name, model in models.items():
        for split_name, (X_eval, y_eval) in datasets.items():
            rows.append(score_model(model, X_eval, y_eval, model_name, split_name, threshold))
    return pd.DataFrame(rows)


models = {
    "Logistic Regression - clean": logistic_model,
    "HistGradientBoosting - clean": hgb_model,
}

evaluation = evaluate_models(models, {"valid": (X_valid, y_valid), "test": (X_test, y_test)})
evaluation.sort_values(["split", "pr_auc"], ascending=[True, False])


## Threshold Analysis

A credit decisioning model needs an operating threshold. The best threshold depends on whether the business wants to maximize approvals, reduce defaults, create a manual-review queue, or optimize a cost function.


In [ ]:
def threshold_table(model, X_eval, y_eval, thresholds=None):
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.55, 0.05), 2)

    rows = []
    proba = model.predict_proba(X_eval)[:, 1]
    for threshold in thresholds:
        pred = (proba >= threshold).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_eval, pred).ravel()
        rows.append({
            "threshold": threshold,
            "precision": precision_score(y_eval, pred, zero_division=0),
            "recall": recall_score(y_eval, pred, zero_division=0),
            "f1": f1_score(y_eval, pred, zero_division=0),
            "flagged_rate": pred.mean(),
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "tp": tp,
        })
    return pd.DataFrame(rows)


best_clean_model = hgb_model
threshold_table(best_clean_model, X_valid, y_valid)


## Lift / Gains Analysis

This is the most business-facing evaluation: if the model ranks applicants from highest to lowest estimated risk, do the highest-risk deciles actually default more often?


In [ ]:
def risk_decile_table(model, X_eval, y_eval, n_bins=10):
    proba = model.predict_proba(X_eval)[:, 1]
    out = pd.DataFrame({"y_true": y_eval, "predicted_pd": proba})
    out["risk_decile"] = pd.qcut(out["predicted_pd"], q=n_bins, labels=False, duplicates="drop") + 1
    out["risk_decile"] = out["risk_decile"].max() + 1 - out["risk_decile"]

    deciles = (
        out.groupby("risk_decile")
        .agg(
            accounts=("y_true", "size"),
            avg_predicted_pd=("predicted_pd", "mean"),
            observed_default_rate=("y_true", "mean"),
            defaults=("y_true", "sum"),
        )
        .reset_index()
        .sort_values("risk_decile")
    )
    deciles["default_capture_rate"] = deciles["defaults"] / deciles["defaults"].sum()
    deciles["cumulative_default_capture"] = deciles["default_capture_rate"].cumsum()
    return deciles


deciles = risk_decile_table(best_clean_model, X_test, y_test)
deciles


In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.plot(deciles["risk_decile"], deciles["observed_default_rate"] * 100, marker="o", label="Observed default rate")
ax1.plot(deciles["risk_decile"], deciles["avg_predicted_pd"] * 100, marker="o", label="Average predicted PD")
ax1.set_xlabel("Risk decile: 1 = highest predicted risk")
ax1.set_ylabel("Rate (%)")
ax1.set_title("Predicted vs observed default rate by risk decile")
ax1.legend()
ax1.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Calibration Check

Calibration matters because credit-risk models often need useful probabilities, not only rankings.


In [ ]:
def calibration_bucket_table(model, X_eval, y_eval, n_bins=10):
    proba = model.predict_proba(X_eval)[:, 1]
    out = pd.DataFrame({"y_true": y_eval, "predicted_pd": proba})
    out["pd_bucket"] = pd.qcut(out["predicted_pd"], q=n_bins, duplicates="drop")
    return (
        out.groupby("pd_bucket", observed=True)
        .agg(
            accounts=("y_true", "size"),
            avg_predicted_pd=("predicted_pd", "mean"),
            observed_default_rate=("y_true", "mean"),
        )
        .reset_index()
    )


calibration_table = calibration_bucket_table(best_clean_model, X_test, y_test)
calibration_table


## Diagnostic Benchmark: Include Pricing / Underwriting Outputs

This benchmark includes `int_rate`, `grade`, and `sub_grade`. It is useful for interpretation, but should not replace the clean origination-only model unless the business confirms these fields are legitimate inputs at the moment the model is used.


In [ ]:
X_pricing, y_pricing, pricing_dates, _ = build_feature_view(
    df,
    include_pricing_outputs=True,
    include_leakage=False,
)

if MODEL_SAMPLE_SIZE is not None and MODEL_SAMPLE_SIZE < len(X_pricing):
    X_pricing = X_pricing.loc[sample_idx]
    y_pricing = y_pricing.loc[sample_idx]
    pricing_dates = pricing_dates.loc[sample_idx]

pricing_splits, pricing_split_summary = chronological_split(X_pricing, y_pricing, pricing_dates)
X_pr_train, y_pr_train = pricing_splits["train"]
X_pr_valid, y_pr_valid = pricing_splits["valid"]
X_pr_test, y_pr_test = pricing_splits["test"]

pricing_model = make_hgb_pipeline(X_pr_train)
pricing_model.fit(X_pr_train, y_pr_train)

pricing_eval = evaluate_models(
    {"HistGradientBoosting - with pricing outputs": pricing_model},
    {"valid": (X_pr_valid, y_pr_valid), "test": (X_pr_test, y_pr_test)},
)

pd.concat([evaluation, pricing_eval], ignore_index=True).sort_values(["split", "pr_auc"], ascending=[True, False])


## Leakage Demonstration

This model intentionally includes invalid post-origination features. It should perform suspiciously well and exists only to demonstrate why leakage controls are essential.


In [ ]:
X_leakage, y_leakage, leakage_dates, _ = build_feature_view(
    df,
    include_pricing_outputs=True,
    include_leakage=True,
)

if MODEL_SAMPLE_SIZE is not None and MODEL_SAMPLE_SIZE < len(X_leakage):
    X_leakage = X_leakage.loc[sample_idx]
    y_leakage = y_leakage.loc[sample_idx]
    leakage_dates = leakage_dates.loc[sample_idx]

leakage_splits, leakage_split_summary = chronological_split(X_leakage, y_leakage, leakage_dates)
X_lk_train, y_lk_train = leakage_splits["train"]
X_lk_valid, y_lk_valid = leakage_splits["valid"]
X_lk_test, y_lk_test = leakage_splits["test"]

leakage_model = make_hgb_pipeline(X_lk_train)
leakage_model.fit(X_lk_train, y_lk_train)

leakage_eval = evaluate_models(
    {"HistGradientBoosting - leakage demo": leakage_model},
    {"valid": (X_lk_valid, y_lk_valid), "test": (X_lk_test, y_lk_test)},
)

pd.concat([evaluation, pricing_eval, leakage_eval], ignore_index=True).sort_values(["split", "pr_auc"], ascending=[True, False])


## Basic Hyperparameter Tuning Plan

After the baseline comparison, tune only the most promising valid model. Keep tuning time-aware and do not tune on the final test period.

Recommended first tuning pass for the gradient boosting model:

- `learning_rate`: `[0.03, 0.06, 0.10]`
- `max_leaf_nodes`: `[15, 31, 63]`
- `l2_regularization`: `[0.0, 0.05, 0.2]`
- `max_iter`: `[100, 200, 300]`

For the presentation, it is enough to show a small, disciplined tuning pass rather than an exhaustive search.


In [ ]:
# Optional compact tuning cell. This intentionally uses the validation set and preserves the final test set.
# Uncomment when you are ready to run the search.

# tuning_rows = []
# for learning_rate in [0.03, 0.06, 0.10]:
#     for max_leaf_nodes in [15, 31, 63]:
#         for l2_regularization in [0.0, 0.05, 0.2]:
#             candidate = make_hgb_pipeline(X_train)
#             candidate.set_params(
#                 model__learning_rate=learning_rate,
#                 model__max_leaf_nodes=max_leaf_nodes,
#                 model__l2_regularization=l2_regularization,
#             )
#             candidate.fit(X_train, y_train)
#             row = score_model(candidate, X_valid, y_valid, "HGB tuned candidate", "valid")
#             row.update({
#                 "learning_rate": learning_rate,
#                 "max_leaf_nodes": max_leaf_nodes,
#                 "l2_regularization": l2_regularization,
#             })
#             tuning_rows.append(row)
#
# tuning_results = pd.DataFrame(tuning_rows).sort_values("pr_auc", ascending=False)
# tuning_results.head(10)


## Model Explanation

For the final presentation, use model explanation to connect predictive performance to credit-risk intuition and governance.

Start with Logistic Regression coefficients for a transparent baseline. Use permutation importance for the selected nonlinear model. SHAP can be added if the package is available and runtime is acceptable.


In [ ]:
def logistic_coefficients_table(model, top_n=30):
    preprocessor = model.named_steps["preprocess"]
    classifier = model.named_steps["model"]
    feature_names = preprocessor.get_feature_names_out()
    coefs = classifier.coef_[0]
    out = pd.DataFrame({"feature": feature_names, "coefficient": coefs})
    out["abs_coefficient"] = out["coefficient"].abs()
    return out.sort_values("abs_coefficient", ascending=False).head(top_n)


logistic_coefficients_table(logistic_model, top_n=30)


In [ ]:
# Permutation importance can be slow on the full test set. Use a sample for a practical notebook runtime.
perm_sample_size = min(20_000, len(X_test))
perm_idx = X_test.sample(perm_sample_size, random_state=RANDOM_STATE).index

perm = permutation_importance(
    best_clean_model,
    X_test.loc[perm_idx],
    y_test.loc[perm_idx],
    n_repeats=5,
    scoring="average_precision",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

perm_importance = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_std": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
)

perm_importance.head(30)


## Final Narrative Checklist

Before presenting, make sure the final story is explicit:

- The model is framed for origination/account-opening default risk.
- Post-origination fields are excluded from the valid model.
- Time-aware validation is the primary evidence.
- SMOTE is not the default because imbalance is moderate; class weighting is the baseline.
- Performance is judged by ranking, calibration, and threshold behavior rather than accuracy alone.
- Explanations and regulatory considerations are included because this is a credit decisioning use case.
